# CulturaQuant — reproduce the null, no GPU

This notebook is an interactive alternative to `python3 reproduce.py`. It re-analyzes the
committed per-item predictions in `data/predictions/` and prints the paper's tables:
differential erosion by quantization, the confidence intervals, and the null result
(int8 culturally safe, nf4 weak/inconsistent). It is **pure Python standard library** —
no GPU, no network, nothing to install. Run the cells top to bottom; total wall-clock is
well under a minute.

In [ ]:
import os, subprocess, sys
# Run from the repo root so the analysis scripts find data/predictions/ by default.
ROOT = os.getcwd()
assert os.path.isdir(os.path.join(ROOT, "analysis")), "run this notebook from the artifact/ root"
PY = sys.executable or "python3"
def run(script):
    print(subprocess.run([PY, os.path.join("analysis", script)],
                          capture_output=True, text=True, check=True).stdout)

## Differential erosion + 95% CIs (the headline)
Per-model and pooled cultural-minus-control drop, with item-clustered bootstrap CIs and the
minimum detectable effect. Expect pooled **int8 +0.2 pp [-0.4, 0.7]**, pooled
**nf4 +0.7 pp [-0.1, 1.6]**, **MDE 1.8 pp** — the int8 null and the nf4 sign-disagreement.

In [ ]:
run("cq_ci.py")

## Position-bias RStd
Spread of chosen-option positions per measurable model. Expect the fp16 RStd range
**0.120 to 0.198** (and the lone high nf4 outlier on Qwen3-1.7B).

In [ ]:
run("cq_rstd.py")

## Measurable set + per-band / per-region drops
Which models clear the five-way chance floor at full precision (**6 of 11**), the full
per-model table, and the checks that cultural drop does not grow with rarity nor concentrate
in one region.

In [ ]:
run("cq_full_analysis.py")

## Full-precision calibration gradient
A sanity check that the probe measures knowledge: full-precision accuracy climbs with item
frequency, from ~21% on the rarest band to ~52% on the most common.

In [ ]:
run("cq_calib.py")

## Verify the macros reproduce byte-for-byte
Regenerate `results/results_macros.tex` from the analysis and assert every `\newcommand`
matches the committed reference. Expect `OK_MACROS_REPRODUCED`.

In [ ]:
print(subprocess.run([PY, "reproduce.py"], capture_output=True, text=True).stdout.splitlines()[-1])